In [1]:
"""
============================================================
 PRINCE Block Cipher — Reversible Quantum Circuit
 Module 1: Pre-whitening layer  (XOR k0, XOR RC0, XOR k1)
============================================================

 Cipher parameters:
   - Plaintext  : 64 bits
   - Key K      : 128 bits  →  split into k0 (high 64) and k1 (low 64)
   - k0'        : derived from k0 via the PRINCE key schedule

 What this module implements (left edge of the block diagram):
   ┌──────────────────────────────────────────────────────┐
   │  state = plaintext ⊕ k0 ⊕ RC0 ⊕ k1                  │
   └──────────────────────────────────────────────────────┘

 Why this is reversible:
   Every XOR is implemented with CNOT (or Pauli-X for fixed
   constants).  Both gates are self-inverse, so the entire
   circuit is its own inverse up to input initialisation.

 Qubit registers (256 qubits total):
   state[0..63]  — 64-bit data path  (plaintext flows in, intermediate
                   state flows out; will carry ciphertext at the end)
   k0[0..63]     — key half k0  (loaded once, reused each time the
                   oracle is queried — in Grover it will be in superposition)
   k1[0..63]     — key half k1  (same note as k0)
   k0p[0..63]    — k0 prime  (pre-computed from k0; needed for the
                   post-whitening step at the very end of PRINCE)

 Bit convention throughout:
   qubit[0]  ↔  bit 0  (LSB) of the corresponding integer
   qubit[63] ↔  bit 63 (MSB)

 Simulation backend:
   Aer 'stabilizer' method.
   The pre-whitening layer uses ONLY Clifford gates (X and CNOT),
   so the stabilizer simulator is exact and runs in polynomial time
   regardless of qubit count.  No statevector explosion.

Dependencies:
   pip install qiskit qiskit-aer
"""

"\n============================================================\n PRINCE Block Cipher — Reversible Quantum Circuit\n Module 1: Pre-whitening layer  (XOR k0, XOR RC0, XOR k1)\n============================================================\n\n Cipher parameters:\n   - Plaintext  : 64 bits\n   - Key K      : 128 bits  →  split into k0 (high 64) and k1 (low 64)\n   - k0'        : derived from k0 via the PRINCE key schedule\n\n What this module implements (left edge of the block diagram):\n   ┌──────────────────────────────────────────────────────┐\n   │  state = plaintext ⊕ k0 ⊕ RC0 ⊕ k1                  │\n   └──────────────────────────────────────────────────────┘\n\n Why this is reversible:\n   Every XOR is implemented with CNOT (or Pauli-X for fixed\n   constants).  Both gates are self-inverse, so the entire\n   circuit is its own inverse up to input initialisation.\n\n Qubit registers (256 qubits total):\n   state[0..63]  — 64-bit data path  (plaintext flows in, intermediate\n          

In [2]:
# ── Standard library ──────────────────────────────────────────────────────────
# (none needed beyond Qiskit)

# ── Qiskit ────────────────────────────────────────────────────────────────────
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator

from Round import quantum_round, classical_round
from Helpers import xor_constant_into_register, xor_register_into_register

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
#  SECTION 1 — PRINCE CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════

# Round constants RC0 .. RC10 are fixed 64-bit values derived from the
# fractional part of π.  They are XOR-ed into the state at the start and
# end of every round (and during the pre/post-whitening step for RC0).
# Source: Borghoff et al., "PRINCE – A Low-Latency Block Cipher", ASIACRYPT 2012.

ROUND_CONSTANTS = [
    0x0000000000000000,   # RC0  — all-zero by design (XOR is identity here)
    0x13198A2E03707344,   # RC1
    0xA4093822299F31D0,   # RC2
    0x082EFA98EC4E6C89,   # RC3
    0x452821E638D01377,   # RC4
    0xBE5466CF34E90C6C,   # RC5  — also called α, used in the middle round
    0x7EF84F78FD955CB1,   # RC6
    0x85840851F1AC43AA,   # RC7
    0xC882D32F25323C54,   # RC8
    0x64A51195E0E3610D,   # RC9
    0xD3B5A399CA0C2399,   # RC10
]

# Convenience alias used in the pre/post-whitening layer
RC0 = ROUND_CONSTANTS[0]   # == 0  (XOR with RC0 adds zero gates to the circuit)

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
#  SECTION 2 — CLASSICAL HELPER: KEY SCHEDULE
# ══════════════════════════════════════════════════════════════════════════════

def prince_key_schedule(K_128bit):
    """
    Derive k0, k1, and k0' from the 128-bit master key K.

    PRINCE key schedule (from the spec):
        k0  = K[127:64]          — upper 64 bits of K
        k1  = K[63:0]            — lower 64 bits of K
        k0' = ROR64(k0, 1) ⊕ (k0 >> 63)

    The ROR (rotate-right-by-1) plus the extra XOR with the top bit
    is equivalent to:
        k0' = (k0 >> 1)          — shift right, dropping bit 0
            | (k0 << 63)         — rotate the dropped bit into position 63
            ^ (k0 >> 63)         — XOR with original bit 63
    All arithmetic is mod 2^64.

    k0' is used only in the post-whitening step at the very end of
    encryption; it is pre-computed here so the quantum register can be
    loaded once and held throughout the circuit.

    Parameters
    ----------
    K_128bit : int   128-bit master key (Python integer)

    Returns
    -------
    k0, k1, k0_prime : (int, int, int)   each a 64-bit Python integer
    """
    MASK64 = (1 << 64) - 1

    k0 = (K_128bit >> 64) & MASK64   # high half
    k1 =  K_128bit        & MASK64   # low half

    # ROR64(k0, 1): bit i of k0 maps to bit (i-1) mod 64 of k0'
    ror1     = ((k0 >> 1) | (k0 << 63)) & MASK64
    k0_prime = ror1 ^ (k0 >> 63)         # XOR with the original MSB

    return k0, k1, k0_prime

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
#  SECTION 4 — PRE-WHITENING CIRCUIT BUILDER
# ══════════════════════════════════════════════════════════════════════════════

def build_prewhitening_circuit(plaintext_64bit, K_128bit):
    """
    Build the reversible quantum circuit for PRINCE pre-whitening:

        state  =  plaintext  ⊕  k0  ⊕  RC0  ⊕  k1

    This corresponds to the very first operations in the PRINCE block
    diagram (left side): the two circles labelled ⊕ before R1.

    The circuit has 256 qubits organised as four 64-qubit registers:

        ┌─────────────────────────────────────────────────────────┐
        │  state  — carries the data (plaintext → intermediate)   │
        │  k0     — first key half (controls XOR into state)      │
        │  k1     — second key half (controls XOR into state)     │
        │  k0p    — k0 prime (stored here, used in post-whitening)│
        └─────────────────────────────────────────────────────────┘

    Gate sequence:
        1. Load plaintext  →  X gates on state  where plaintext bit = 1
        2. Load k0         →  X gates on k0     where k0 bit = 1
        3. Load k1         →  X gates on k1     where k1 bit = 1
        4. Load k0'        →  X gates on k0p    where k0' bit = 1
        5. state ^= k0     →  64 CNOT gates  (k0  → state)
        6. state ^= RC0    →  X gates on state for set bits of RC0
                              (RC0 = 0, so this adds ZERO gates)
        7. state ^= k1     →  64 CNOT gates  (k1  → state)
        8. Measure state   →  64 measure operations into classical register

    Parameters
    ----------
    plaintext_64bit : int   64-bit plaintext
    K_128bit        : int   128-bit master key

    Returns
    -------
    qc       : QuantumCircuit   the assembled circuit (with measurement)
    k0       : int              64-bit k0 (for external verification)
    k1       : int              64-bit k1 (for external verification)
    k0_prime : int              64-bit k0' (for external verification)
    """

    # ── Derive key halves ─────────────────────────────────────────────────────
    k0, k1, k0_prime = prince_key_schedule(K_128bit)

    # ── Allocate quantum registers ────────────────────────────────────────────
    state = QuantumRegister(64, name='state')   # data path
    anc   = QuantumRegister(16 * 14, name='anc')
    qk0   = QuantumRegister(64, name='k0')      # key half 0
    qk1   = QuantumRegister(64, name='k1')      # key half 1
    qk0p  = QuantumRegister(64, name='k0p')     # k0 prime (post-whitening)

    # Classical register captures the 64-bit state after pre-whitening
    creg  = ClassicalRegister(64, name='out')

    qc = QuantumCircuit(state, anc, qk0, qk1, qk0p, creg)

    # ── Step 1: Initialise state register with plaintext ─────────────────────
    qc.barrier(label="Init: plaintext → state")
    for bit in range(64):
        if (plaintext_64bit >> bit) & 1:
            qc.x(state[bit])

    # ── Step 2: Initialise k0 register ───────────────────────────────────────
    qc.barrier(label="Init: k0 → k0 register")
    for bit in range(64):
        if (k0 >> bit) & 1:
            qc.x(qk0[bit])

    # ── Step 3: Initialise k1 register ───────────────────────────────────────
    qc.barrier(label="Init: k1 → k1 register")
    for bit in range(64):
        if (k1 >> bit) & 1:
            qc.x(qk1[bit])

    # ── Step 4: Initialise k0' register ──────────────────────────────────────
    qc.barrier(label="Init: k0' → k0p register")
    for bit in range(64):
        if (k0_prime >> bit) & 1:
            qc.x(qk0p[bit])

    # ── Step 5: state ^= k0 ──────────────────────────────────────────────────
    qc.barrier(label="XOR k0 into state")
    xor_register_into_register(qc, qk0, state)

    # ── Step 6: state ^= RC0 ─────────────────────────────────────────────────
    qc.barrier(label="XOR RC0 into state")
    xor_constant_into_register(qc, state, RC0)   # no-op because RC0 == 0

    # Round 1 
    for i in range(1,6):
        quantum_round(i,  qc, state, anc, qk1, ROUND_CONSTANTS[i])
    
    # ── Step 8: Measure the state register ───────────────────────────────────
    qc.barrier(label="Measure state")
    qc.measure(state, creg)

    return qc, k0, k1, k0_prime

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
#  SECTION 5 — CLASSICAL REFERENCE IMPLEMENTATION
# ══════════════════════════════════════════════════════════════════════════════

def classical_prewhitening(plaintext_64bit, K_128bit):
    """
    Pure-Python reference for PRINCE pre-whitening.
    Used to cross-check the quantum circuit output.

        state = plaintext ⊕ k0 ⊕ RC0 ⊕ k1

    Parameters
    ----------
    plaintext_64bit : int   64-bit plaintext
    K_128bit        : int   128-bit master key

    Returns
    -------
    state    : int   64-bit intermediate state after pre-whitening
    k0       : int   64-bit k0
    k1       : int   64-bit k1
    k0_prime : int   64-bit k0'
    """
    MASK64 = (1 << 64) - 1
    k0, k1, k0_prime = prince_key_schedule(K_128bit)

    state = plaintext_64bit & MASK64
    state ^= k0
    state ^= RC0    # == 0, so state unchanged; included for structural clarity

    for i in range(1,6):
        state = classical_round(i, state, k1, ROUND_CONSTANTS[i])
    
    return state, k0, k1, k0_prime

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
#  SECTION 6 — SIMULATION HELPER
# ══════════════════════════════════════════════════════════════════════════════

def bitstring_to_int(bitstring):
    """
    Convert an Aer measurement bitstring to a Python integer.

    Aer returns classical register bits MSB-first:
        bitstring[0]  = creg[63]  = state qubit 63  = integer bit 63
        bitstring[63] = creg[0]   = state qubit 0   = integer bit 0

    Therefore  int(bitstring, 2)  gives the correct integer directly,
    because int() treats the leftmost character as the most significant bit.

    Example:
        bitstring = '1111111011011100...'  (64 chars)
        int(bitstring, 2) == 0xFEDCBA9876543210   ✓

    Parameters
    ----------
    bitstring : str   64-character '0'/'1' string from Aer counts

    Returns
    -------
    int   64-bit integer
    """
    return int(bitstring, 2)


def simulate(qc, sim):
    result = sim.run(qc, shots=1).result()
    counts = result.get_counts()
    return int(list(counts.keys())[0], 2)

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
#  SECTION 7 — MAIN: BUILD, SIMULATE, AND VERIFY
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    # Aer stabilizer backend — handles any number of Clifford-only qubits
    # exactly, in polynomial time.  We reuse the same simulator object
    # across all tests.
    sim = AerSimulator(method='matrix_product_state')

    # ──────────────────────────────────────────────────────────────────────────
    #  TEST 1 — All-zero key and plaintext
    #  Expected: state = 0 ⊕ 0 ⊕ 0 ⊕ 0 = 0x0000000000000000
    # ──────────────────────────────────────────────────────────────────────────
    print("=" * 62)
    print("  TEST 1 — All-zero plaintext and key")
    print("=" * 62)

    pt1 = 0x0000000000000000
    K1  = 0x00000000000000000000000000000000

    # Classical reference
    ref1, k0_1, k1_1, k0p_1 = classical_prewhitening(pt1, K1)
    print(f"  Plaintext          : 0x{pt1:016X}")
    print(f"  Key K              : 0x{K1:032X}")
    print(f"  k0 (K[127:64])     : 0x{k0_1:016X}")
    print(f"  k1 (K[63:0])       : 0x{k1_1:016X}")
    print(f"  k0' (ROR(k0,1)^msb): 0x{k0p_1:016X}")
    print(f"  RC0                : 0x{RC0:016X}")
    print(f"  Classical result   : 0x{ref1:016X}")

    # Quantum circuit
    qc1, _, _, _ = build_prewhitening_circuit(pt1, K1)
    meas1 = simulate(qc1, sim)

    print(f"  Quantum result     : 0x{meas1:016X}")
    print(f"  Match              : {'✅ PASS' if meas1 == ref1 else '❌ FAIL'}")

    # ──────────────────────────────────────────────────────────────────────────
    #  TEST 2 — Non-zero key and plaintext
    #  k0 = pt  →  k0 ⊕ pt = 0  →  state = k1 = 0xFEDCBA9876543210
    # ──────────────────────────────────────────────────────────────────────────
    print()
    print("=" * 62)
    print("  TEST 2 — Non-zero plaintext and key")
    print("=" * 62)

    pt2 = 0x1111111111111111
    K2  = 0x0ABCDEFFEDC123456789BA9876543210

    ref2, k0_2, k1_2, k0p_2 = classical_prewhitening(pt2, K2)
    print(f"  Plaintext          : 0x{pt2:016X}")
    print(f"  Key K              : 0x{K2:032X}")
    print(f"  k0 (K[127:64])     : 0x{k0_2:016X}")
    print(f"  k1 (K[63:0])       : 0x{k1_2:016X}")
    print(f"  k0' (ROR(k0,1)^msb): 0x{k0p_2:016X}")
    print(f"  RC0                : 0x{RC0:016X}")
    print(f"  Classical result   : 0x{ref2:016X}")

    qc2, _, _, _ = build_prewhitening_circuit(pt2, K2)
    meas2 = simulate(qc2, sim)

    print(f"  Quantum result     : 0x{meas2:016X}")
    print(f"  Match              : {'✅ PASS' if meas2 == ref2 else '❌ FAIL'}")

    # ──────────────────────────────────────────────────────────────────────────
    #  TEST 3 — Reversibility
    #  The XOR operations (state^=k0, state^=RC0, state^=k1) must be
    #  reversible.  Applying them then un-applying them must return the
    #  state to the value it had after initialisation (i.e. the plaintext).
    #
    #  Strategy: build a circuit with the initialisation (X gates) followed
    #  by the XOR computation twice.  The second pass is the inverse of the
    #  first, so the net effect on state must be identity → state == plaintext.
    # ──────────────────────────────────────────────────────────────────────────
    print()
    print("=" * 62)
    print("  TEST 3 — Reversibility  (forward ∘ inverse = identity)")
    print("=" * 62)

    # We build the circuit in two parts:
    #   Part A — initialisation (load plaintext, k0, k1, k0')
    #   Part B — XOR computation only (state^=k0, state^=RC0, state^=k1)
    # Reversibility test: run Part A, then Part B, then Part B again.
    # Since Part B is self-inverse (all Clifford), state must equal plaintext.

    k0_3, k1_3, k0p_3 = prince_key_schedule(K2)

    state3 = QuantumRegister(64, name='state')
    qk0_3  = QuantumRegister(64, name='k0')
    qk1_3  = QuantumRegister(64, name='k1')
    qk0p_3 = QuantumRegister(64, name='k0p')
    creg3  = ClassicalRegister(64, name='rev')
    qc3    = QuantumCircuit(state3, qk0_3, qk1_3, qk0p_3, creg3)

    # Part A: initialise all registers (same as build_prewhitening_circuit)
    qc3.barrier(label="Init: plaintext")
    for bit in range(64):
        if (pt2 >> bit) & 1:
            qc3.x(state3[bit])
    qc3.barrier(label="Init: k0")
    for bit in range(64):
        if (k0_3 >> bit) & 1:
            qc3.x(qk0_3[bit])
    qc3.barrier(label="Init: k1")
    for bit in range(64):
        if (k1_3 >> bit) & 1:
            qc3.x(qk1_3[bit])
    qc3.barrier(label="Init: k0p")
    for bit in range(64):
        if (k0p_3 >> bit) & 1:
            qc3.x(qk0p_3[bit])

    # Part B (forward): state ^= k0 ^= RC0 ^= k1
    qc3.barrier(label="Forward: XOR k0")
    xor_register_into_register(qc3, qk0_3, state3)
    qc3.barrier(label="Forward: XOR RC0")
    xor_constant_into_register(qc3, state3, RC0)
    qc3.barrier(label="Forward: XOR k1")
    xor_register_into_register(qc3, qk1_3, state3)

    # Part B (inverse): apply the SAME gates again in REVERSE order.
    # For CNOTs and X gates  g^{-1} = g, so we just replay them in
    # reverse order: state ^= k1 ^= RC0 ^= k0
    qc3.barrier(label="Inverse: XOR k1")
    xor_register_into_register(qc3, qk1_3, state3)
    qc3.barrier(label="Inverse: XOR RC0")
    xor_constant_into_register(qc3, state3, RC0)
    qc3.barrier(label="Inverse: XOR k0")
    xor_register_into_register(qc3, qk0_3, state3)

    # Measure — state must equal the original plaintext
    qc3.barrier(label="Measure")
    qc3.measure(state3, creg3)

    meas3 = simulate(qc3, sim)

    print(f"  Input plaintext              : 0x{pt2:016X}")
    print(f"  After forward + inverse      : 0x{meas3:016X}")
    print(f"  Reversible (should match pt) : {'✅ PASS' if meas3 == pt2 else '❌ FAIL'}")

    # 
    print("=" * 62)
    print("  TEST 4 — Non-zero plaintext and key")
    print("=" * 62)

    pt2 = 0x15325031ABFE3210
    K2  = 0x0ABCDE2581B123456782FF5876543210

    ref2, k0_2, k1_2, k0p_2 = classical_prewhitening(pt2, K2)
    print(f"  Plaintext          : 0x{pt2:016X}")
    print(f"  Key K              : 0x{K2:032X}")
    print(f"  k0 (K[127:64])     : 0x{k0_2:016X}")
    print(f"  k1 (K[63:0])       : 0x{k1_2:016X}")
    print(f"  k0' (ROR(k0,1)^msb): 0x{k0p_2:016X}")
    print(f"  RC0                : 0x{RC0:016X}")
    print(f"  Classical result   : 0x{ref2:016X}")

    qc2, _, _, _ = build_prewhitening_circuit(pt2, K2)
    meas2 = simulate(qc2, sim)

    print(f"  Quantum result     : 0x{meas2:016X}")
    print(f"  Match              : {'✅ PASS' if meas2 == ref2 else '❌ FAIL'}")

    
    # ──────────────────────────────────────────────────────────────────────────
    #  CIRCUIT RESOURCE REPORT
    # ──────────────────────────────────────────────────────────────────────────
    print()
    print("=" * 62)
    print("  CIRCUIT RESOURCE REPORT  (test-2 circuit)")
    print("=" * 62)

    print(f"  Qubit registers:")
    for reg in qc2.qregs:
        print(f"    {reg.name:8s}  {reg.size} qubits")

    ops = dict(qc2.count_ops())
    print(f"  Total qubits      : {qc2.num_qubits}")
    print(f"  Classical bits    : {qc2.num_clbits}")
    print(f"  Circuit depth     : {qc2.depth()}")
    print(f"  X  gate count     : {ops.get('x', 0)}")
    print(f"  CNOT gate count   : {ops.get('cx', 0)}")
    print(f"  Measure count     : {ops.get('measure', 0)}")
    print(f"  Toffoli (ccx)     : {ops.get('ccx', 0)}")
    print()

  TEST 1 — All-zero plaintext and key
  Plaintext          : 0x0000000000000000
  Key K              : 0x00000000000000000000000000000000
  k0 (K[127:64])     : 0x0000000000000000
  k1 (K[63:0])       : 0x0000000000000000
  k0' (ROR(k0,1)^msb): 0x0000000000000000
  RC0                : 0x0000000000000000
  Classical result   : 0x5DA5C90D56459BFB
  Quantum result     : 0x5DA5C90D56459BFB
  Match              : ✅ PASS

  TEST 2 — Non-zero plaintext and key
  Plaintext          : 0x1111111111111111
  Key K              : 0x0ABCDEFFEDC123456789BA9876543210
  k0 (K[127:64])     : 0x0ABCDEFFEDC12345
  k1 (K[63:0])       : 0x6789BA9876543210
  k0' (ROR(k0,1)^msb): 0x855E6F7FF6E091A2
  RC0                : 0x0000000000000000
  Classical result   : 0xECF8B8D31BEC4ABD
  Quantum result     : 0xECF8B8D31BEC4ABD
  Match              : ✅ PASS

  TEST 3 — Reversibility  (forward ∘ inverse = identity)
  Input plaintext              : 0x1111111111111111
  After forward + inverse      : 0x11111111111111

  Quantum result     : 0x82266A6BCFF95727
  Match              : ✅ PASS

  TEST 2 — Non-zero plaintext and key
  Plaintext          : 0x1111111111111111
  Key K              : 0x0ABCDEFFEDC123456789BA9876543210
  k0 (K[127:64])     : 0x0ABCDEFFEDC12345
  k1 (K[63:0])       : 0x6789BA9876543210
  k0' (ROR(k0,1)^msb): 0x855E6F7FF6E091A2
  RC0                : 0x0000000000000000
  Classical result   : 0x57CD19657EF088BD


  Quantum result     : 0x57CD19657EF088BD
  Match              : ✅ PASS

  TEST 3 — Reversibility  (forward ∘ inverse = identity)
  Input plaintext              : 0x1111111111111111
  After forward + inverse      : 0x1111111111111111
  Reversible (should match pt) : ✅ PASS
  TEST 4 — Non-zero plaintext and key
  Plaintext          : 0x15325031ABFE3210
  Key K              : 0x0ABCDE2581B123456782FF5876543210
  k0 (K[127:64])     : 0x0ABCDE2581B12345
  k1 (K[63:0])       : 0x6782FF5876543210
  k0' (ROR(k0,1)^msb): 0x855E6F12C0D891A2
  RC0                : 0x0000000000000000
  Classical result   : 0xB3ED3FD115974173


  Quantum result     : 0xB3ED3FD115974173
  Match              : ✅ PASS

  CIRCUIT RESOURCE REPORT  (test-2 circuit)
  Qubit registers:
    state     64 qubits
    anc       224 qubits
    k0        64 qubits
    k1        64 qubits
    k0p       64 qubits
  Total qubits      : 480
  Classical bits    : 64
  Circuit depth     : 1794
  X  gate count     : 323
  CNOT gate count   : 2400
  Measure count     : 64
  Toffoli (ccx)     : 1280

